# 3. Evaluating a run: every table and plot

`evaluate()` takes run folders and returns the scores as tables. `lfe evaluate` is the same function, and also writes the tables and the report to `results/<name>/`: `report.html`, and a page for every form.

To have something to evaluate, this notebook first makes four small runs with the oracles of `examples/oracle_extractor.py` (they read the gold, so they are not submissions): a clean and a sloppy one on SpokenWOZ, and the same on AMI. `docs/metrics.md` explains every number.

In [ ]:
import sys

import pandas as pd

from live_form_eval import evaluate, load_config, run
from live_form_eval.config import REPO_ROOT
from live_form_eval.evaluation.report import plots

sys.path.insert(0, str(REPO_ROOT))
from examples.oracle_extractor import Oracle, SloppyOracle

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.3f}'.format)
config = load_config()
mode = config.plots.mode

# development runs: a few conversations, replayed 30 times faster than real time (so they are unofficial)
woz = [run(extractor(), 'spokenwoz', config=config, split='dev', n=15, speed=30, concurrency=15) for extractor in (Oracle, SloppyOracle)]
ami = [run(extractor(), 'ami', config=config, n=4, speed=30, concurrency=4) for extractor in (Oracle, SloppyOracle)]

## evaluate()

Give it one run, one run per dataset of the same submission, or the runs of several submissions to compare them (here: the two oracles). AMI's free text is judged by a language model that runs locally, on a GPU when there is one (`docs/kaggle.md`). The first time, this cell takes a few minutes. The judge keeps its verdicts in `results/.cache`, so texts it has already judged cost nothing the next time.

In [ ]:
from live_form_eval.evaluation.evaluate import judge_store
from live_form_eval.evaluation.matching.judge.nli import load_judge

# loaded once and reused below, with the folder its verdicts are kept in; evaluate() does the same when no judge is given
judge = load_judge(config.judge, store=judge_store(config))
result = evaluate(woz + ami, config=config, judge=judge, write=False)
print(result.summary_text())

## The tables

`result.summary` has one row per run, with every metric.

In [ ]:
result.summary.drop(columns=['run_dir', 'run']).set_index(['name', 'dataset']).T

`result.ranking` has one row per submission: its score over the datasets it ran (the weighted average of its runs' scores), the four terms behind it and the price per form. Official submissions come first, best score on top. These runs are unofficial.

In [ ]:
result.ranking

`result.per_field_type` and `result.per_field` break the final form, completeness, precision, speed and latency down. They are for understanding a result, not for ranking.

In [ ]:
sloppy_woz = result.summary[(result.summary['name'] == 'SloppyOracle') & (result.summary['dataset'] == 'spokenwoz')].iloc[0]
latency = f'latency_p{config.latency.percentile:g}_s'
result.per_field_type[result.per_field_type['run'] == sloppy_woz['run']][
    ['field_type', 'correct', 'wrong', 'missing', 'invented', 'correctly_empty', 'final_f1', 'completeness', 'precision', 'speed', 'latency_median_s', latency, 'never_settled']]

`result.values` has one row per value, option or item of every final form: the gold, what was shown, the outcome, when it was spoken (`spoken_s`, and where that time comes from), when its words reached the extractor (`available_s`), and its latency, counted from `available_s`.

In [ ]:
values = result.values[result.values['run'] == sloppy_woz['run']]
values[values['outcome'] != 'correctly_empty'][['conversation', 'field_id', 'label', 'shown', 'outcome', 'spoken_s', 'available_s', 'said_from', 'latency_s']].head(14)

## The plots

Each function returns an Altair chart; `plots.themed` applies the colours. A plot of several runs takes a key: which runs, under which names and in which colours. `plots.run_key` makes the one the report uses, for all the runs or for those of one dataset, so a run has the same colour in every plot. Hover a mark for its numbers.

### Score against price

Every run is a point, in one plot per dataset: what a form costs, against its score. The upper left is where to be: a high score for a cheap form. The oracles report no price, so they sit on the left edge.

In [ ]:
everyone = plots.run_key(result.summary, mode)   # every run: its name in the report, and its colour
plots.themed(plots.score_against_price(result.summary, everyone, mode), mode)

### The runs of a dataset next to each other

The score and its four terms for the two oracles on SpokenWOZ, the highest score first.

In [ ]:
woz_key = plots.run_key(result.summary, mode, 'spokenwoz')   # the runs of one dataset
plots.themed(plots.terms_by_run(result.summary, woz_key, mode), mode)

### By field type (SpokenWOZ)

Final F1 is the form at the end. The three terms are read while people are talking: was each value shown, was what stood on screen right, and how soon? The second plot counts the final form's values: correct, missing, invented and wrong, one plot per run.

In [ ]:
plots.themed(plots.by_field_type(result.per_field_type, woz_key, mode), mode)

In [ ]:
plots.themed(plots.outcomes_by_field_type(result.per_field_type, woz_key, mode), mode)

### Final Decision latency

The share of values that were on screen for good within x seconds of their words reaching the extractor, one line per run. It is reported next to the score, not ranked. The sloppy oracle never shows 10% of its values, so its line stays below 90%.

In [ ]:
plots.themed(plots.latency_curves(result.values, woz_key, mode), mode)

### One conversation, field by field

Each field has a grey line, named after the field: what it should show, one bar per gold step, from the moment its words reached the extractor. Under it, green, amber and red: what the extractor showed, coloured by its grade (right, close, wrong), with a mark at every change of the field (● first fill, ▲ revision, ✕ cleared). This is the picture behind completeness, precision and speed, and the fastest way to see what to fix.

The report of an evaluation has this plot on the page of every form, for all its runs at once (see "The report" at the end).

In [ ]:
busiest = values[values['outcome'] != 'correctly_empty'].groupby('conversation').size().idxmax()
plots.conversation_timeline(sloppy_woz['run_dir'], busiest, config)

## AMI: free text

Notes and answers go through the same walk as typed fields. Their gold values are sets of **facts**: single statements, each from the moment the meeting has said all of it. A judge compares a text with them in both directions: does the text say every fact (complete), and do the facts support everything the text says (correct)? The similarity is the harmonic mean of the two, and a text equals a gold value when both are 1. Unanswerable questions must stay empty. `docs/metrics.md`, "Free text (AMI)", has the details.

The sloppy oracle leaves out 10% of the facts: its notes stay right (precision of the final form 1) but incomplete.

In [ ]:
ami_rows = result.summary[result.summary['dataset'] == 'ami'].set_index('name')
ami_rows[['score', 'completeness', 'precision', 'speed', 'outcome', 'free_text_correct', 'free_text_complete', 'free_text_f1',
          'unanswerable_accuracy', 'final_f1', 'fixes_per_form', 'latency_median_s', latency, 'never_settled', 'judge']].T

In [ ]:
sloppy_ami = ami_rows.loc['SloppyOracle']
result.per_field_type[result.per_field_type['run'] == sloppy_ami['run']][
    ['field_type', 'correct', 'missing', 'invented', 'correctly_empty', 'final_f1', 'completeness', 'precision', 'speed', 'latency_median_s', 'never_settled']]

In [ ]:
ami_key = plots.run_key(result.summary, mode, 'ami')
plots.themed(plots.latency_curves(result.values, ami_key, mode), mode)

In [ ]:
missed = result.values[(result.values['run'] == sloppy_ami['run']) & result.values['outcome'].isin(['missing', 'invented'])]
missed[['conversation', 'field_id', 'unit', 'label', 'outcome']].head(10)

### One meeting, field by field

The same picture as for a call, with the judge. Grey: the facts a note or an answer should hold, one bar per gold step. Green, amber and red: each text the field showed, coloured by its grade. Hover a bar to read the facts or the text.

In [ ]:
meetings = result.per_conversation[result.per_conversation['run'] == sloppy_ami['run']]
lowest = meetings.sort_values('score').iloc[0]['conversation']   # the meeting where the sloppy oracle lost most
plots.conversation_timeline(sloppy_ami['run_dir'], lowest, config, judge=judge)

## The report

With `write=True` (the default, and what `lfe evaluate` does), the tables are written as CSV files next to the report:

- `report.html`: the ranking and the score against the price, then one section per dataset with all its runs in the same tables and plots: the scores, the field types (the plots above), the model calls with their tokens and their price (for runs that report calls; the oracles make none), and how often values changed.
- `forms/`: a page for every conversation, chosen at the top of the report. It shows the scores of each run on that form, the timeline above for all the runs at once, the model calls each run made on it, the form at the end next to the gold, every field's values over time, the transcript with the gold each turn completes, and the form itself (its rules and all its fields).

Open `report.html` in a browser. The runs given to one `evaluate()` share every table, plot and page, each run in its own colour (here the two oracles). From the command line, `uv run lfe evaluate results/<a> results/<b>` puts the runs of earlier results into one report in the same way.

In [ ]:
written = evaluate(woz + ami, config=config, judge=judge, out=config.paths.results / 'notebook-03')
print(sorted(path.name for path in written.out.iterdir()))
pages = sorted(path.name for path in (written.out / 'forms').iterdir())
print(len(pages), 'form pages:', ', '.join(pages[:4]), '…')
print('open', written.out / 'report.html')